# Programmieraufgabe 5: Arnoldi-Prozess, GMRES und FOM

**Abgabe in den Programmiertutorien am 16./17. Juli 2025.**

Benötigte Module:

In [ ]:
import numpy as np
import scipy.linalg as spla
import matplotlib.pyplot as plt
import numpy.random as rnd

## 1. Der Arnoldi-Prozess

**(a) Schreiben Sie eine Methode `arnoldi(A, b, M)`, die wie in der Vorlesung vorgestellt eine Orthonormalbasis des $M$-ten Krylov-Raum $\mathcal{K}_M(A,b)$ berechnet. Verwenden Sie dabei die modifizierte Variante des Arnoldi-Algorithmus. Die Funktion soll sowohl die Matrix $V_M$ (deren Spalten die Vektoren der Orthogonalbasis sind), als auch die "Hessenberg-Matrix" $\widetilde{H}_M\in\mathbb{R}^{(M+1) \times M}$ zurückgeben.**

Hinweise: 
* Es genügt, Ihre Funktion für Matrizen $A$ und Vektoren $b$ mit reellen Einträgen zu schreiben.
* Statt $h_{m+1,m}=0$ zu überprüfen, sollten Sie besser das Kriterium $|h_{m+1,m}|<10^{-14}$ verwenden.
* Falls der Abbruch für ein $m<M$ eintritt, dann sollen die Matrizen $V_m$ und $\widetilde{H}_m$ (anstatt $V_M$ und $\widetilde{H}_M$) zurückgegeben und ein entsprechender Hinweis auf dem Bildschirm ausgegeben werden.

Testen Sie an den folgenden beiden Beispielen, ob Ihre Funktion das tut, was sie soll. 

In [ ]:
# Matrix A und Vektor b:
n = 10
A = rnd.random([n,n])
b = rnd.random([n,])

# Wende Arnoldi mit gewünschtem M an
M = n
V, H = arnoldi(A, b, M)

# Welche Dimension hat die erhaltene Basis tatsächlich?
M = V.shape[1]
print('Zur Info: Dimension des erhaltenen Krylov-Raums:',M )
# Gilt AV_{M-1} = V_M \widetilde{H}_{M-1}?
print('Wenn die Arnoldi-Relation gilt, sollte folgende Norm (quasi) Null sein:')
print( spla.norm(A@V[:,:-1] - V@H[:-1,:-1]) )
# Ist die Basis orthonormal?
print('Wenn V eine ONB enthält, sollte folgende Norm (quasi) Null sein:')
print( spla.norm(V.T@V- np.eye(M)) )

In [ ]:
# Matrix A und Vektor b:
A = np.array([[0,0,1,1],[0,2,0,0],[1,0,0,1],[1,1,-1,1]])
b = np.array([2,1,2,1])
# Wende Arnoldi mit gewünschtem M an
M = n
V, H = arnoldi(A, b, M)

# Welche Dimension hat die erhaltene Basis tatsächlich?
M = V.shape[1]
print('Zur Info: Dimension des erhaltenen Krylov-Raums:',M )
# Gilt AV_{M-1} = V_M \widetilde{H}_{M-1}?
print('Wenn die Arnoldi-Relation gilt, sollte folgende Norm (quasi) Null sein:')
print( spla.norm(A@V[:,:-1] - V@H[:-1,:-1]) )
# Ist die Basis orthonormal?
print('Wenn V eine ONB enthält, sollte folgende Norm (quasi) Null sein:')
print( spla.norm(V.T@V- np.eye(M)) )

Nun scheint es so, als hätten wir eine gute Orthonormalbasis des ($M$-ten) Krylov-Raums bestimmt. Eigentlich möchten wir aber die Lösung des Gleichungssystems $Ax = b$ approximieren. Das machen wir jetzt als nächstes - und zwar mit dem FOM und dem GMRES-Verfahren.

## 2. GMRES- und FOM-Verfahren

Bei beiden Verfahren werden nicht direkt Approximationen $x_m \approx \hat{x} := A^{-1} b$ berechnet, sondern zunächst Vektoren $y_m\in\mathbb{R}^m$, welche die Koeffizienten der Approximationen $x_m$ bezüglich der in $V_m$ gespeicherten Basis enthalten. Die Approximationen $x_m$ erhält man dann also über $x_m = V_m y_m$. Wir kümmern uns daher zuerst um die Prozeduren, die uns die Koeffizientenvektoren $y_m$ der GMRES-/FOM-Approximationen liefern.

**(b) Schreiben Sie eine Funktion `gmres(H, b)`, die zu gegebener Matrix $\tilde{H}_m\in\mathbb{R}^{(m+1)\times m}$ und rechter Seite $b\in\mathbb{R}^{n}$ die Lösung des linearen Ausgleichsproblems
$$  \|\tilde{H}_m y - \beta e_1\|_2 = \min_{y\in\mathbb{R}^n} ! $$
berechnet und diese zurückgibt. Dabei gilt $\beta = \|b\|_2$, und $e_1$ ist der erste Einheitsvektor in $\mathbb{R}^{m+1}$.** 

*Sie dürfen dabei die Funktion `spla.lstsq(...)` verwenden. (Daher wird Ihre Funktion `gmres(H, b)` vermutlich sehr kurz sein...)*

**(c) Schreiben Sie eine Funktion `fom(H, b)`, die zu gegebener Matrix ${H}_m\in\mathbb{R}^{m\times m}$ und rechter Seite $b\in\mathbb{R}^{n}$ die Lösung des linearen Gleichungssystems
$$  {H}_m y = \beta e_1$$
berechnet und diese zurückgibt. Dabei gilt $\beta = \|b\|_2$, und $e_1$ ist der erste Einheitsvektor in $\mathbb{R}^{m}$.** 

*Sie dürfen dabei die Funktion `spla.solve(...)` verwenden. (Daher wird Ihre Funktion `FOM(H, b)` vermutlich sehr kurz sein...)*

**(d) Mit der folgenden Code-Zelle sollen Sie Ihre Prozeduren testen. Dazu müssen Sie sie passend ergänzen: Verwenden Sie zunächst die Prozedur `arnoldi` mit $M=n$, und berechnen Sie dann die Näherungslösungen von GMRES und FOM. Da wegen $M=n$ die exakte Lösung auf jeden Fall im berechneten Krylovraum enthalten ist, sollten `gmres` und `fom` die exakte Lösung liefern. Dies wird im restlichen Teil des Code-Blocks anhand einer mit dem LGS-Löser aus `scipy` berechneten Referenzlösung überprüft.**

*Hinweis: Natürlich stimmen die über GMRES und FOM berechneten Näherungslösungen nur bis auf Rundungsfehler mit der exakten Lösung $\hat{x}$ überein. Daher wird der relative Fehler überprüft. Wenn dieser im Bereich $10^{-14}$ oder kleiner ist, dann haben Sie vermutlich alles richtig gemacht.*

In [ ]:
n = 10
A = rnd.random([n,n])
b = rnd.random([n,])
xhat = spla.solve(A,b)

# hier sind Sie gefragt...
# ... TODO
# ... TODO
# ... TODO
x_gmres = # ... TODO
x_fom   = # ... TODO

# GMRES
err_gmres = spla.norm(x_gmres - xhat) / spla.norm(xhat) 
print('Relativer Fehler GMRES:',err_gmres)
# FOM
err_fom = spla.norm(x_fom - xhat) / spla.norm(xhat) 
print('Relativer Fehler FOM:',err_fom)

## 3. Konvergenz von FOM und GMRES?

Zum Schluss wollen wir beobachten, in wiefern die Approximationen von GMRES und FOM für größer werdende Krylov-Räume konvergieren. Aus der Vorlesung kennen wir (noch) keine erwartete "Konvergenzrate", stattdessen wollen wir einfach exemplarisch beobachten, was passiert.

Wir betrachten dazu zwei verschiedene Matrizen:
1. Eine zufällige Matrix (dementsprechend mit zufälligen Eigenwerten)
2. Eine positiv definite Matrix, bei der alle Eigenwerte mindestens $\alpha$ für ein $\alpha\geq0$ sind.

**(e) Ergänzen Sie die untenstehende Prozedur `create_error_plot` an den mit `TODO` markierten Stellen so, dass die Approximation $x_m \in \mathcal{K}_m(A,b)$ des GMRES- und des FOM-Verfahrens sowie die relativen Fehler
$$
\frac{\lVert x_m - \widehat{x}\rVert}{\lVert\widehat{x}\rVert} 
\qquad \text{und} \qquad
\frac{\lVert Ax_m - b\rVert}{\lVert b \rVert} = \frac{\lVert r_m \rVert}{\lVert b \rVert}
$$ der Approximationen bzw. der Residuen für alle $m=1,...,n$ berechnet und die Fehler in einem Plot dargestellt werden.**

Um den Fehler der Approximationen berechnen zu können, ist natürlich die Kenntnis der exakten Lösung erforderlich. Diese berechnen wir hier mit einem direkten `scipy`-LGS-Löser.

In [ ]:
def create_error_plot(A,b):
    n = A.shape[0]
    print('Kondition der Matrix:',np.linalg.cond(A))

    # Referenzlösung
    xhat = spla.solve(A,b)

    # Arnoldi-Algorithmus mit m=N
    V, H = arnoldi(A, b, n)

    # tatsächlich Dimension des finalen Krylov-Raums:
    M = H.shape[1]

    # Bereite Fehlervektoren vor (für FOM und GMRES):
    # relativer Fehler der Lösung
    err_gmres = np.zeros([M,])
    err_fom = np.zeros([M,])
    # relativer Fehler der Residuen
    res_gmres = np.zeros([M,])
    res_fom = np.zeros([M,])

    for m in range(0,M):
        # Approximation des GMRES-Verfahren im m+1-ten Krylov-Raum, m=0,...,M-1
        # ... TODO
        x_gmres = # ... TODO
        err_gmres[m] = spla.norm(x_gmres - xhat) / spla.norm(xhat)
        res_gmres[m] = spla.norm(A@x_gmres - b) / spla.norm(b) 
        # Approximation des FOM-Verfahren im m+1-ten Krylov-Raum, m=0,...,M-1
        y # ... TODO
        x_fom = # ... TODO
        err_fom[m] = spla.norm(x_fom - xhat) / spla.norm(xhat) 
        res_fom[m] = spla.norm(A@x_fom - b) / spla.norm(b) 

    # Plot vorbereiten
    fig, (ax1, ax2) = plt.subplots(1, 2,figsize=(9, 4))
    mvec = range(1,M+1)

    # Fehler GMRES im ersten Subplot plotten
    ax1.plot(mvec, err_gmres,
             mvec, res_gmres)
    ax1.legend(["Fehler Approximation","Fehler Residuum"])
    ax1.set_title('GMRES')
    ax1.set_xlabel('Dimension Krylov-Raum m')

    # Fehler FOM im ersten Subplot plotten
    ax2.plot(mvec, err_fom,
             mvec, res_fom)
    ax2.legend(["Fehler Approximation","Fehler Residuum"])
    ax2.set_title('FOM')
    ax2.set_xlabel('Dimension Krylov-Raum m')

**(f) Wenden Sie die Prozedur `create_error_plot` auf die oben beschriebenen Matrizen an, indem Sie die zwei folgenden Zellen ausführen. Führen Sie jede Zelle mehrfach aus, um zu sehen, wie sich das Bild zwischen verschiedenen zufälligen Matrizen ändern kann. Sie sollten unter Anderem sehen, dass eine der vier Fehlerkurven immer monoton fallend ist. Warum ist das so?** 

In [ ]:
# Zufällige Matrix mit Einträgen in [0,1]
n = 20
A = rnd.random([n,n])
b = rnd.random([n,])

create_error_plot(A,b)

In [ ]:
# Zufällige symmetrisch postiv definite Matrix mit Eigenwerten >= alpha
n = 20
A = rnd.random((n, n))
alpha = 0.1 # testen Sie auch alpha = 0, 0.1, 1, 5
A = A@A.T + alpha*np.eye(n)
b = rnd.random((n,))

create_error_plot(A,b)

*Hinweise:*
* Für den letzten Fall werden wir in einer zukünftigen Tutoriums- oder Hausaufgabe eine gewisse Konvergenzrate des GMRES-Verfahrens beweisen können.
* In diesem Notebook wurde keine sehr effiziente Realisierung der GMRES- und FOM-Verfahren entwickelt. Insbesondere würde man die Orthonormalbasis der Krylov-Räume iterativ ergänzen und im selben Schritt auch die Approximation der Lösung des LGS berechnen - und zwar so lange, bis man eine *gut genuge* Approximation erhält (was auch immer das heißt). Dass wir anders vorgegangen sind, hat hauptsächlich didaktische Gründe (und ist zudem etwas leichter zu debuggen). 